# Machine-learning evaluation: a hands-on implementation

**Track:** Machine learning · **Guide:** [`008-ml-evaluation.md`](../guides/concepts/008-ml-evaluation.md)

Model evaluation estimates generalization with data kept separate from fitting and uses metrics tied to the real cost of errors.

> **Intuition:** A fair exam uses unseen questions and a scoring rubric that reflects which mistakes matter.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Machine-learning evaluation]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Choose a decision threshold for a rare but costly equipment failure instead of relying on accuracy alone.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Separate development from final testing

Use validation data for choices and keep the test set untouched until the final assessment.


In [ ]:
from sklearn.model_selection import train_test_split

X = [[value] for value in range(100)]
y = [int(value >= 70) for value in range(100)]
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=14, stratify=y
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=14, stratify=y_trainval
)
print("train:", len(X_train), "validation:", len(X_validation), "test:", len(X_test))

## 2. Inspect class-sensitive metrics

For rare failures, recall may matter more than overall accuracy; inspect the confusion matrix and precision-recall trade-off.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
validation_predictions = baseline.predict(X_validation)
print(confusion_matrix(y_validation, validation_predictions))
print(classification_report(y_validation, validation_predictions, zero_division=0))
cv_recall = cross_val_score(
    baseline, X_train, y_train, cv=3, scoring="recall"
)
print("training-only cross-validation recall:", cv_recall.round(2).tolist())

## 3. Tune a threshold on validation data

Pick a threshold to match operational costs using validation data, then report once on untouched test data.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
validation_probabilities = model.predict_proba(X_validation)[:, 1]
threshold = 0.4
thresholded_validation = (validation_probabilities >= threshold).astype(int)
final_model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_trainval, y_trainval)
test_predictions = (final_model.predict_proba(X_test)[:, 1] >= threshold).astype(int)
print("validation positives:", int(thresholded_validation.sum()))
print("final test confusion matrix:")
print(confusion_matrix(y_test, test_predictions))

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/008-ml-evaluation.md) and the
[learning roadmap](../guides/ROADMAP.md).
